# バイオ技術 2-2：画像AIとCNNの基礎

このNotebookでは、手書き数字画像のMNISTデータセットを使って、
**Convolutional Neural Network（CNN）による画像分類**を体験します。

2-1では、Decision Treeを使って、

> どの条件分岐を通って予測されたか

を確認しました。

2-2ではCNNを使い、

> 画像AIがどのように学習し、どのように予測し、どの画像領域が予測に関係したか

を見ていきます。

---

## 今日のゴール

このNotebookが終わるころには、次のことを説明できることを目指します。

1. コンピュータにとって画像は数値の集まりである
2. CNNが画像の特徴を学習するモデルである
3. Convolution、Pooling、Dense層のおおまかな役割
4. Training / Validation / Test dataの違い
5. 予測結果を「各クラスの確率」として確認できる
6. Confusion Matrixや誤分類画像からモデルの弱点を確認できる
7. Grad-CAMを使って、予測に関係した領域を可視化できる

---

## このNotebookの進め方

途中で、画像番号や設定値を自分で変更します。

**実行する → 結果を見る → 値を変える → 比較する → 考える**

を意識して進めてください。


---
## 0. 画像分類AIとは？

画像分類では、

> **1枚の画像を入力し、その画像がどのクラスに属するかを予測する**

という問題を扱います。

今回のMNISTでは、

- 入力：28 × 28ピクセルの手書き数字画像
- 出力：0〜9のどの数字か

を予測します。

大まかな流れは、

```text
Image
  ↓
CNN
  ↓
Score for class 0, 1, 2, ..., 9
  ↓
Probability
  ↓
Prediction
```

です。


---
## 1. 必要なライブラリを読み込む

Google ColabにはTensorFlow / Kerasがあらかじめ用意されています。
今回は追加インストールせず、そのまま使用します。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)


---
## 2. MNISTデータセットを読み込む

MNISTは、0〜9の手書き数字画像からなる代表的な画像データセットです。

今回は、

- Training data：モデルの学習に使う
- Test data：学習後の性能評価に使う

という役割で使います。


In [ ]:
from tensorflow.keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape :", x_test.shape)
print("y_test shape :", y_test.shape)


### データの形を読む

例えば、

```text
x_train shape: (60000, 28, 28)
```

なら、

- 60,000枚の画像
- 1枚あたり28 × 28ピクセル

という意味です。

ラベルは0〜9の整数です。


In [ ]:
print("Unique labels:", np.unique(y_train))


### ミニ確認

- Training画像数：
- Test画像数：
- 画像サイズ：
- クラス数：

を記録してください。


---
## 3. まず画像を人間の目で見る

Training dataから1枚表示します。


In [ ]:
# ↓↓↓ 自分で数字を変更してください ↓↓↓
IMAGE_INDEX = 0

plt.figure(figsize=(3, 3))
plt.imshow(x_train[IMAGE_INDEX], cmap="gray")
plt.title(f"Label = {y_train[IMAGE_INDEX]}")
plt.axis("off")
plt.show()


### ミニ演習1：別の画像を見る

`IMAGE_INDEX`を、

- 10
- 100
- 1000

などに変更してください。

### 考えてみよう

- 同じ数字でも書き方は同じですか？
- 人間には読めても、AIには難しそうな画像はありますか？


---
## 4. コンピュータには画像がどう見えているか

人間には数字の形に見えますが、コンピュータには**数値の行列**として見えています。

まず1枚目の画像を数値として確認します。


In [ ]:
print("Image shape:", x_train[0].shape)
print()
print(x_train[0])


画素値は0〜255です。

- 0：暗い
- 255：明るい

次に、画像の一部分だけを表として見ます。


In [ ]:
small_region = x_train[0][8:18, 8:18]

display(
    pd.DataFrame(small_region)
)


### 考えてみよう

人間は「5」という形を見ています。

一方、コンピュータは、

> どの位置に、どの程度の値があるか

を扱っています。

画像では、単一の画素だけでなく、**近くの画素同士の配置や形**が重要です。  
この考え方がCNNにつながります。


---
## 5. 複数の画像を見る

ランダムに16枚表示します。


In [ ]:
rng = np.random.default_rng(42)
indices = rng.choice(
    len(x_train),
    size=16,
    replace=False
)

plt.figure(figsize=(8, 8))

for i, idx in enumerate(indices):
    plt.subplot(4, 4, i + 1)
    plt.imshow(x_train[idx], cmap="gray")
    plt.title(f"Label: {y_train[idx]}")
    plt.axis("off")

plt.tight_layout()
plt.show()


---
## 6. 前処理：0〜1に正規化する

MNIST画像の画素値は0〜255です。

255で割り、

> 0〜255 → 0〜1

に変換します。

またCNNに入力するため、

```text
28 × 28
```

を

```text
28 × 28 × 1
```

にします。

最後の`1`は、グレースケール画像のチャンネル数です。


In [ ]:
X_train = x_train.astype("float32") / 255.0
X_test = x_test.astype("float32") / 255.0

X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("Original shape :", x_train.shape)
print("CNN input shape:", X_train.shape)

print()
print("Pixel range after scaling:")
print("min =", X_train.min())
print("max =", X_train.max())


---
## 7. CNNとは？

CNNはConvolutional Neural Networkの略です。

画像のように、

> **近くにある情報同士の関係が重要なデータ**

に適したニューラルネットワークです。

今回のCNNは次の流れです。

```text
Input image
    ↓
Convolution
    ↓
Pooling
    ↓
Convolution
    ↓
Pooling
    ↓
Flatten
    ↓
Dense
    ↓
10-class output
```


### Convolution層

Convolution層では、小さなフィルタを画像上で動かしながら特徴を抽出します。

初期の層では、

- 線
- 角
- 曲線

など、比較的単純な特徴を捉えます。

後ろの層では、それらを組み合わせたより複雑な特徴を扱います。

重要なのは、

> **どの特徴を抽出するかを人間がすべて指定するのではなく、学習によってフィルタの重みが調整される**

ことです。


### Pooling層

Pooling層では、特徴マップを小さくします。

主な目的は、

- 計算量を減らす
- 特徴をまとめる
- 少しの位置ずれに対して頑健にする

ことです。


### FlattenとDense層

ConvolutionとPoolingの出力は、まだ2次元の特徴マップです。

`Flatten`で1次元に変換し、`Dense`層で最終的な分類を行います。

最後の出力は10個あり、0〜9の10クラスに対応します。


---
## 8. CNNモデルを作る

今回は、比較的小さなCNNをFunctional APIで作ります。

Functional APIでは、

> 入力 → 各層 → 出力

を明示的につなぎます。

また、Convolution層では`padding="same"`を使います。

これにより、Convolutionの前後で空間的な位置対応を保ちやすくなり、
あとでGrad-CAMを元画像に重ねるときも、注目領域をより自然に対応させやすくなります。

この書き方にすると、後でGrad-CAMを作るときに、
途中の畳み込み層`conv2`と最終出力を同時に取り出せます。


In [ ]:
inputs = keras.Input(
    shape=(28, 28, 1),
    name="input_image"
)

x = layers.Conv2D(
    filters=16,
    kernel_size=(3, 3),
    padding="same",
    activation="relu",
    name="conv1"
)(inputs)

x = layers.MaxPooling2D(
    pool_size=(2, 2),
    name="pool1"
)(x)

x = layers.Conv2D(
    filters=32,
    kernel_size=(3, 3),
    padding="same",
    activation="relu",
    name="conv2"
)(x)

x = layers.MaxPooling2D(
    pool_size=(2, 2),
    name="pool2"
)(x)

x = layers.Flatten(
    name="flatten"
)(x)

x = layers.Dense(
    64,
    activation="relu",
    name="dense1"
)(x)

# softmax前の値（logits）を出力する
outputs = layers.Dense(
    10,
    name="classifier"
)(x)

model = keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="mnist_cnn"
)

model.summary()


### `model.summary()`を見るポイント

- Inputは28 × 28 × 1
- `padding="same"`なので、Convolution層では縦横サイズを保つ
- Pooling層で縦横サイズが小さくなる
- 最後は10個の値を出力する

### 考えてみよう

1. `conv1`の出力shapeは？
2. `pool1`の後で画像サイズはどう変わりましたか？
3. `conv2`の出力shapeは？
4. 最後の出力が10なのはなぜですか？

答え：

1.
2.
3.
4.


---
## 9. 学習できる状態にする

今回は、

- optimizer：Adam
- loss：Sparse Categorical Crossentropy
- metric：Accuracy

を使います。

最後の層はsoftmax前の値（logits）を出力するので、
`from_logits=True`を指定します。

予測確率を見たいときは、あとでsoftmaxを適用します。


In [ ]:
model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"]
)


---
## 10. CNNを学習する

今回は演習時間を考え、5 epochs学習します。

`validation_split=0.1`により、Training dataの一部をValidation dataとして使います。

- Training data：重みを更新する
- Validation data：学習中の性能を確認する
- Test data：最終評価に使う


In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)


---
## 11. 学習曲線を見る

学習中のAccuracyとLossの変化を確認します。


In [ ]:
history_df = pd.DataFrame(history.history)
display(history_df)


In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(
    history_df.index + 1,
    history_df["accuracy"],
    marker="o",
    label="train"
)

plt.plot(
    history_df.index + 1,
    history_df["val_accuracy"],
    marker="o",
    label="validation"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Learning Curve: Accuracy")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(
    history_df.index + 1,
    history_df["loss"],
    marker="o",
    label="train"
)

plt.plot(
    history_df.index + 1,
    history_df["val_loss"],
    marker="o",
    label="validation"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning Curve: Loss")
plt.legend()
plt.show()


### ミニ演習2：学習曲線を読む

1. Epochが進むとTraining accuracyはどうなりましたか？
2. Validation accuracyも同じように変化しましたか？
3. Training performanceだけ見れば十分でしょうか？

答え：

1.
2.
3.


---
## 12. Test dataで性能を評価する

学習に使っていないTest dataで評価します。


In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")


---
## 13. 1枚の画像を予測する

Test dataから1枚選びます。

`TEST_IMAGE_INDEX`を書き換えて、好きな画像を選んでください。


In [ ]:
# ↓↓↓ 0〜9999の範囲で変更してください ↓↓↓
TEST_IMAGE_INDEX = 0

selected_image = X_test[
    TEST_IMAGE_INDEX:TEST_IMAGE_INDEX + 1
]

logits = model.predict(
    selected_image,
    verbose=0
)[0]

probabilities = tf.nn.softmax(
    logits
).numpy()

predicted_label = int(
    np.argmax(probabilities)
)

true_label = int(
    y_test[TEST_IMAGE_INDEX]
)

plt.figure(figsize=(3, 3))
plt.imshow(
    x_test[TEST_IMAGE_INDEX],
    cmap="gray"
)
plt.title(
    f"True: {true_label}   Predicted: {predicted_label}"
)
plt.axis("off")
plt.show()

print("True label     :", true_label)
print("Predicted label:", predicted_label)


### ミニ演習3：別の画像を予測する

`TEST_IMAGE_INDEX`を変更して、少なくとも3枚試してください。

| Index | True | Predicted | Correct? |
|---|---|---|---|
| | | | |
| | | | |
| | | | |


---
## 14. 予測確率を見る

AIは数字を1つだけ返しているわけではありません。

0〜9それぞれについてスコアを出し、
softmaxによって確率に変換できます。


In [ ]:
probability_df = pd.DataFrame({
    "digit": np.arange(10),
    "probability": probabilities
})

display(probability_df)


In [ ]:
plt.figure(figsize=(8, 4))

plt.bar(
    probability_df["digit"],
    probability_df["probability"]
)

plt.xticks(np.arange(10))
plt.xlabel("Digit")
plt.ylabel("Probability")
plt.title(
    f"Prediction Probabilities: index {TEST_IMAGE_INDEX}"
)
plt.show()


### 考えてみよう

次の2種類の画像を探してみてください。

1. 1つの数字の確率が非常に高い画像
2. 複数の数字に確率が分かれている画像

- 自信が高そうなImage Index：
- 迷っていそうなImage Index：

予測確率を見ると、AIの「迷い方」の手がかりが得られます。


---
## 15. Grad-CAMで予測に関係した領域を見る

2-1のDecision Treeでは、

> どの条件分岐を通って判定したか

を確認しました。

CNNはDecision Treeより複雑で、判断過程をそのまま人間が追うことは困難です。

そこで、**Grad-CAM**を使います。

Grad-CAMでは、

> あるクラスの予測スコアに対して、最後の畳み込み層のどの領域が強く関係したか

をヒートマップとして可視化します。

### 注意

Grad-CAMで強調された領域は、

- AIがそこだけを見た
- その領域が原因である

ことを証明するものではありません。

**予測に関係した領域を理解するための手がかり**として使います。


### Grad-CAM用の関数を準備する

今回は、最後の畳み込み層`conv2`を使います。


In [ ]:
LAST_CONV_LAYER_NAME = "conv2"

def make_gradcam_heatmap(
    img_array,
    model,
    last_conv_layer_name,
    class_index=None
):
    # 最終畳み込み層の出力と最終出力を同時に取得するモデル
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.outputs[0]
        ]
    )

    with tf.GradientTape() as tape:
        conv_outputs, output_logits = grad_model(
            img_array,
            training=False
        )

        if class_index is None:
            class_index = int(
                tf.argmax(output_logits[0]).numpy()
            )

        class_score = output_logits[:, class_index]

    grads = tape.gradient(
        class_score,
        conv_outputs
    )

    # 各チャンネルの重要度
    weights = tf.reduce_mean(
        grads,
        axis=(0, 1, 2)
    )

    # 1枚目の特徴マップ
    conv_outputs = conv_outputs[0]

    # 重み付き和
    heatmap = tf.reduce_sum(
        conv_outputs * weights,
        axis=-1
    )

    # 正の寄与のみ残す
    heatmap = tf.maximum(
        heatmap,
        0
    )

    # 0〜1へ正規化
    max_value = tf.reduce_max(heatmap)
    heatmap = heatmap / (
        max_value + keras.backend.epsilon()
    )

    return heatmap.numpy()


### 選択中の画像でGrad-CAMを表示する

先ほど`TEST_IMAGE_INDEX`で選んだ画像について、
**予測クラスに対するGrad-CAM**を表示します。

左から、

1. 元画像
2. Grad-CAM
3. 元画像への重ね合わせ

です。


### Overlayの表示について

Heatmapそのものは0〜1の連続値で表示します。

Overlayでは、弱い反応まで色を重ねると元画像が見えにくくなるため、
このNotebookでは`OVERLAY_THRESHOLD = 0.30`未満の領域を透明にします。

この閾値は見やすさのための表示上の設定であり、
Grad-CAMの計算結果そのものを変更するものではありません。


In [ ]:
selected_heatmap = make_gradcam_heatmap(
    selected_image,
    model,
    LAST_CONV_LAYER_NAME,
    class_index=predicted_label
)

# Grad-CAMを元画像サイズへ拡大
selected_heatmap_resized = tf.image.resize(
    selected_heatmap[..., np.newaxis],
    (28, 28)
).numpy().squeeze()

# Overlayでは弱い反応を透明にする
OVERLAY_THRESHOLD = 0.30

selected_overlay = selected_heatmap_resized.copy()
selected_overlay[
    selected_overlay < OVERLAY_THRESHOLD
] = np.nan

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(
    x_test[TEST_IMAGE_INDEX],
    cmap="gray"
)
plt.title(
    f"Original\nTrue={true_label}, Pred={predicted_label}"
)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(
    selected_heatmap_resized,
    cmap="jet",
    vmin=0,
    vmax=1
)
plt.title("Grad-CAM")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(
    x_test[TEST_IMAGE_INDEX],
    cmap="gray"
)
plt.imshow(
    selected_overlay,
    cmap="jet",
    alpha=0.55,
    vmin=0,
    vmax=1
)
plt.title(
    f"Overlay\nthreshold={OVERLAY_THRESHOLD}"
)
plt.axis("off")

plt.tight_layout()
plt.show()


### ミニ演習4：Grad-CAMを観察する

`TEST_IMAGE_INDEX`を変更して、いくつかの画像を比較してください。

Overlayでは、弱い反応を透明にして、比較的強い反応領域を見やすくしています。

次の点を考えてみましょう。

1. 数字のどの部分が強く強調されていますか？
2. 背景ではなく、数字の線上が強調されていますか？
3. 画像によって注目領域は変わりますか？
4. 予測確率が高い画像と、迷っている画像で違いはありますか？

記録：

- Image Index：
- True：
- Predicted：
- Grad-CAMで強調された領域：


---
## 16. Test data全体を予測する


In [ ]:
test_logits = model.predict(
    X_test,
    batch_size=256,
    verbose=0
)

test_probabilities = tf.nn.softmax(
    test_logits,
    axis=1
).numpy()

test_predictions = np.argmax(
    test_probabilities,
    axis=1
)

print("Number of predictions:", len(test_predictions))


---
## 17. Confusion Matrixを見る

2-1では乳がん分類のConfusion Matrixを見ました。

今回は10クラスなので、

> どの数字を、どの数字と間違えやすいか

を確認できます。


In [ ]:
cm = confusion_matrix(
    y_test,
    test_predictions
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=np.arange(10)
)

fig, ax = plt.subplots(figsize=(9, 9))

disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

plt.title("MNIST Confusion Matrix")
plt.show()


### ミニ演習5：間違えやすい組み合わせを探す

1. どの数字同士が間違えられていますか？
2. 人間から見ても形が似ていますか？

答え：

-
-


---
## 18. 誤分類画像を見る

AIが間違えた画像を実際に見てみます。


In [ ]:
wrong_indices = np.where(
    test_predictions != y_test
)[0]

print("Number of misclassified images:", len(wrong_indices))
print("First 20 wrong indices:")
print(wrong_indices[:20])


In [ ]:
N_SHOW = 16

plt.figure(figsize=(10, 10))

for i, idx in enumerate(wrong_indices[:N_SHOW]):
    plt.subplot(4, 4, i + 1)
    plt.imshow(
        x_test[idx],
        cmap="gray"
    )
    plt.title(
        f"True:{y_test[idx]} Pred:{test_predictions[idx]}"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()


### ミニ演習6：AIの間違いを観察する

1. 人間にも読みにくい画像はありますか？
2. AIの予測も理解できそうな画像はありますか？
3. 人間には簡単そうなのに、AIが間違えている画像はありますか？
4. Test accuracyが高くても、失敗例を見ることは重要だと思いますか？

答え：

1.
2.
3.
4.


---
## 19. 誤分類画像を1つ選んで詳しく見る

`WRONG_NUMBER`を変更して、誤分類例を1つ調べます。


In [ ]:
# ↓↓↓ 0〜len(wrong_indices)-1 の範囲で変更してください ↓↓↓
WRONG_NUMBER = 0

idx = int(
    wrong_indices[WRONG_NUMBER]
)

true_wrong = int(
    y_test[idx]
)

pred_wrong = int(
    test_predictions[idx]
)

probs_wrong = test_probabilities[idx]

plt.figure(figsize=(3, 3))
plt.imshow(
    x_test[idx],
    cmap="gray"
)
plt.title(
    f"True: {true_wrong}   Predicted: {pred_wrong}"
)
plt.axis("off")
plt.show()

detail_df = pd.DataFrame({
    "digit": np.arange(10),
    "probability": probs_wrong
}).sort_values(
    "probability",
    ascending=False
)

display(detail_df)


### 考えてみよう

- 正解：
- 予測：
- 2番目に確率が高い数字：
- AIは強く間違えていますか？それとも迷っていますか？
- 画像を見ると、その理由を想像できますか？


---
## 20. 誤分類画像のGrad-CAMを比較する

同じ誤分類画像について、

- **予測クラス**に対するGrad-CAM
- **正解クラス**に対するGrad-CAM

を比較します。

CNNが誤分類したとき、それぞれのクラスのスコアに関係した領域を見比べてみます。


In [ ]:
wrong_image = X_test[idx:idx + 1]

# 予測クラスに対するGrad-CAM
pred_heatmap = make_gradcam_heatmap(
    wrong_image,
    model,
    LAST_CONV_LAYER_NAME,
    class_index=pred_wrong
)

# 正解クラスに対するGrad-CAM
true_heatmap = make_gradcam_heatmap(
    wrong_image,
    model,
    LAST_CONV_LAYER_NAME,
    class_index=true_wrong
)

pred_heatmap_resized = tf.image.resize(
    pred_heatmap[..., np.newaxis],
    (28, 28)
).numpy().squeeze()

true_heatmap_resized = tf.image.resize(
    true_heatmap[..., np.newaxis],
    (28, 28)
).numpy().squeeze()

# Overlayでは弱い反応を透明にする
OVERLAY_THRESHOLD = 0.30

pred_overlay = pred_heatmap_resized.copy()
pred_overlay[
    pred_overlay < OVERLAY_THRESHOLD
] = np.nan

true_overlay = true_heatmap_resized.copy()
true_overlay[
    true_overlay < OVERLAY_THRESHOLD
] = np.nan

plt.figure(figsize=(16, 7))

# 元画像
plt.subplot(2, 3, 1)
plt.imshow(
    x_test[idx],
    cmap="gray"
)
plt.title(
    f"Original\nTrue={true_wrong}, Pred={pred_wrong}"
)
plt.axis("off")

# 予測クラス heatmap
plt.subplot(2, 3, 2)
plt.imshow(
    pred_heatmap_resized,
    cmap="jet",
    vmin=0,
    vmax=1
)
plt.title(
    f"Grad-CAM\nPredicted class: {pred_wrong}"
)
plt.axis("off")

# 正解クラス heatmap
plt.subplot(2, 3, 3)
plt.imshow(
    true_heatmap_resized,
    cmap="jet",
    vmin=0,
    vmax=1
)
plt.title(
    f"Grad-CAM\nTrue class: {true_wrong}"
)
plt.axis("off")

# 予測クラス overlay
plt.subplot(2, 3, 5)
plt.imshow(
    x_test[idx],
    cmap="gray"
)
plt.imshow(
    pred_overlay,
    cmap="jet",
    alpha=0.55,
    vmin=0,
    vmax=1
)
plt.title(
    f"Overlay\nPredicted class: {pred_wrong}"
)
plt.axis("off")

# 正解クラス overlay
plt.subplot(2, 3, 6)
plt.imshow(
    x_test[idx],
    cmap="gray"
)
plt.imshow(
    true_overlay,
    cmap="jet",
    alpha=0.55,
    vmin=0,
    vmax=1
)
plt.title(
    f"Overlay\nTrue class: {true_wrong}"
)
plt.axis("off")

plt.tight_layout()
plt.show()


### ミニ演習7：誤分類の理由を考える

予測クラスと正解クラスについて、HeatmapとOverlayを比較してください。

1. 予測クラスでは、数字のどの部分が強調されていますか？
2. 正解クラスでは、強調領域がどのように違いますか？
3. 予測された数字らしく見える部分はありますか？
4. 同じ画像でも、対象クラスによってGrad-CAMが変化することを確認できましたか？
5. Grad-CAMだけで誤分類の理由を完全に説明できたと言えるでしょうか？

答え：

1.
2.
3.
4.
5.


---
## 21. Explainable AI（XAI）

AIモデルの予測根拠を理解しやすくする方法を、広く**Explainable AI（XAI）**と呼びます。

今回の2日目では、

- 2-1：Decision Treeの条件分岐を読む
- 2-2：CNNのGrad-CAMを見る

という2種類の「説明可能性」を体験しました。

ただし、可視化結果を過信せず、

> **モデルの判断を理解するための補助情報として扱う**

ことが重要です。


---
## 22. まとめ

このNotebookでは、MNISTを使ってCNNによる画像分類とGrad-CAMによる可視化を体験しました。

### 今日の流れ

1. MNIST画像を見る
2. 画像を数値行列として見る
3. 0〜1へ正規化する
4. CNNモデルを作る
5. CNNを学習する
6. 学習曲線を見る
7. Test dataで評価する
8. 1枚の画像を予測する
9. 予測確率を見る
10. Grad-CAMで予測に関係した領域を見る
11. Grad-CAMを元画像に重ねて位置関係を確認する
12. Confusion Matrixを見る
13. 誤分類画像を観察する
14. 誤分類画像について、予測クラスと正解クラスのGrad-CAMを比較する

---

## 最後の確認

**Q1. コンピュータにとって画像とは何ですか？**

答え：

**Q2. CNNのConvolution層は何をする層ですか？**

答え：

**Q3. 予測確率を見ると何がわかりますか？**

答え：

**Q4. Grad-CAMは何を可視化する方法ですか？**

答え：

**Q5. Grad-CAMで強調された領域は、AIの判断理由を完全に証明するものですか？**

答え：

**Q6. Accuracyが高くても誤分類画像を見る意味は何だと思いますか？**

答え：

---

## 次のNotebookへ

2-1では表形式データの分類、2-2では画像分類を行いました。

次の2-3では、

> **糖尿病データを使った回帰問題**

に挑戦します。

2-3では穴埋め問題を増やし、自分でコードを完成させながら解析します。
